# 🧠 LONQ — Sistema de Detección de Régimen + Señales de Trading

**Sistema multicapa de análisis cuantitativo:**

| Capa | Módulo | Qué hace |
|------|--------|----------|
| 1 | **Ensemble de Régimen** | KMeans + GMM + HMM → consenso bull/sideways/bear |
| 2 | **Trigger Multi-TF** | Detecta movimientos anómalos (diario/semanal/quincenal/mensual) |
| 3 | **Overlay VIX** | Ajusta el régimen con el miedo macro del mercado |
| 4 | **Señal Combinada** | Integra las 3 capas → BUY_DIP / TAKE_PROFIT / HOLD con precio objetivo |

---
## ✅ Cómo usar este notebook

1. **Ejecuta la celda `📦 Setup`** — solo la primera vez por sesión
2. **Edita la celda `⚙️ Config`** — escribe tu ticker (ej. `"AAPL"`)
3. **Ejecuta todas las celdas restantes** (`Runtime → Run all` o `Ctrl+F9`)
4. Al final verás el **Estado Actual**: régimen, VIX, señal vigente y niveles de alerta

> **Tickers pre-descargados** (más rápido, sin internet): `KO` `TSLA` `SPY` `BTC-USD`  
> **Cualquier otro ticker** se descarga automáticamente vía yfinance


In [ ]:
# ╔══════════════════════════════════════════════╗
# ║  📦  SETUP — ejecutar una vez por sesión     ║
# ╚══════════════════════════════════════════════╝
import subprocess, sys, os

print('📦 Instalando dependencias...')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'hmmlearn>=0.3.0', 'scikit-learn', 'pandas',
    'numpy', 'scipy', 'matplotlib', 'yfinance'
], capture_output=True)

REPO   = '/content/A_tecn'
BRANCH = 'claude/continue-lonq-7VADd'
REPO_URL = 'https://github.com/pfreezv/A_tecn.git'

if not os.path.exists(REPO):
    print('📥 Clonando repositorio LONQ...')
    r = subprocess.run(
        ['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO],
        capture_output=True, text=True
    )
    print('✅ Clonado' if r.returncode == 0 else f'❌ Error: {r.stderr[:200]}')
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--quiet'], capture_output=True)
    print('✅ Repositorio actualizado')

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from IPython.display import display, HTML

print(f'📁 Directorio: {os.getcwd()}')
print('✅ Todo listo — edita la celda Config y ejecuta el resto')

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║   ⚙️  CONFIGURACIÓN — SOLO EDITAR ESTA CELDA        ║
# ╚══════════════════════════════════════════════════════╝

TICKER     = 'KO'          # ← PON AQUÍ TU TICKER
                            #   Ej: 'AAPL'  'MSFT'  'NVDA'  'AMZN'  'SPY'
                            #   Pre-descargados: 'KO'  'TSLA'  'SPY'  'BTC-USD'

START_DATE = '2018-01-01'  # Inicio del histórico
USE_VIX    = True          # True = incluir overlay de miedo macro

# ── Parámetros avanzados (no hace falta cambiarlos) ────
THRESHOLD  = 2.0   # Z-score para anomalías (2.0 → 95% días normales)
HORIZON    = 10    # Días hacia adelante para precio objetivo
MIN_SCORE  = 2     # Puntuación mínima para emitir señal (1–5)
K_MIN, K_MAX = 2, 6  # Rango de clusters

# ═══════════════════════════════════════════════════════
print(f'⚙️  Configurado: {TICKER}  |  VIX={USE_VIX}  |  σ={THRESHOLD}  |  horizonte={HORIZON}d')

In [ ]:
# ── 📥 CARGA DE DATOS ──────────────────────────────────
CSV_MAP = {
    'KO':      'ko_data.csv',
    'TSLA':    'tsla_data.csv',
    'SPY':     'spy_data.csv',
    'BTC-USD': 'btc_data.csv',
    'BTC':     'btc_data.csv',
}

def load_ohlcv(ticker, start_date):
    t = ticker.upper()
    if t in CSV_MAP and os.path.exists(CSV_MAP[t]):
        raw = pd.read_csv(CSV_MAP[t], index_col='Date', parse_dates=True)
        src = f'CSV local ({CSV_MAP[t]})'
    else:
        try:
            import yfinance as yf
            raw = yf.download(t, start=start_date, progress=False, auto_adjust=True)
            if raw.empty:
                raise ValueError('sin datos')
            raw.columns = [c[0] if isinstance(c, tuple) else c for c in raw.columns]
            src = 'yfinance'
        except Exception as e:
            raise RuntimeError(
                f'No se pudo obtener datos para {t}.\n'
                f'Opciones: usa un ticker pre-descargado (KO/TSLA/SPY/BTC-USD)\n'
                f'o sube un CSV con columnas Date,Open,High,Low,Close,Volume.\nError: {e}'
            )
    raw.index = pd.to_datetime(raw.index).tz_localize(None)
    raw = raw[raw.index >= start_date].dropna(subset=['Close'])
    return raw, src

print(f'📥 Cargando {TICKER}...')
raw, data_src = load_ohlcv(TICKER, START_DATE)
prices = raw['Close']

# VIX
vix_series = None
if USE_VIX:
    from src.vix import get_vix
    vix_series = get_vix(
        path='vix_data.csv',
        start_date=START_DATE,
        max_age_days=5,    # actualiza si tiene más de 5 días de antigüedad
        auto_update=True,  # descarga de Yahoo Finance si es necesario
    )
    if vix_series is not None:
        common_vix = len(prices.index.intersection(vix_series.index))
        print(f'😱 VIX listo — {len(vix_series)} días totales, {common_vix} en común con {TICKER}')
        print(f'   Último dato VIX: {vix_series.index[-1].date()}  ({vix_series.iloc[-1]:.1f})')
    else:
        print('⚠️  VIX no disponible — continuando sin overlay')

# Resumen
vol_ann   = prices.pct_change().std() * np.sqrt(252)
ret_total = prices.iloc[-1] / prices.iloc[0] - 1
print(f"""
{'═'*52}
  📊 {TICKER.upper()} — Datos cargados
{'═'*52}
  Fuente:        {data_src}
  Período:       {prices.index[0].date()} → {prices.index[-1].date()}
  Días:          {len(prices)}
  Precio:        ${prices.iloc[0]:.2f} → ${prices.iloc[-1]:.2f}
  Retorno total: {ret_total:+.1%}
  Vol. anual:    {vol_ann:.1%}
{'═'*52}""")

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  🧠  CAPA 1 — ENSEMBLE DE RÉGIMEN                   ║
# ║      KMeans + GMM + HMM → consenso bull/side/bear   ║
# ╚══════════════════════════════════════════════════════╝
from src.ensemble import fit_ensemble

print(f'\n{"═"*58}')
print(f'  🧠 CAPA 1 — ENSEMBLE DE RÉGIMEN  ({TICKER})')
print(f'  KMeans + GMM + HMM  |  K={K_MIN}–{K_MAX}')
print(f'{"═"*58}\n')

ensemble = fit_ensemble(TICKER, raw, k_min=K_MIN, k_max=K_MAX, show_progress=True)

# ── Indicadores de adecuación ───────────────────────────
def _rate(v):
    if v is None or (hasattr(v,'__float__') and __import__('math').isnan(float(v))): return '—   n/a'
    v = float(v)
    if v > 0.40: return f'{v:.3f}  ✓✓ Excelente'
    if v > 0.30: return f'{v:.3f}  ✓  Bueno'
    if v > 0.20: return f'{v:.3f}  ~  Aceptable'
    return     f'{v:.3f}  ⚠  Pobre'

m   = ensemble.metrics
sils = [float(m.loc[k,'Sil_Train']) for k in ['K-Means','GMM','HMM']]
avg_sil  = __import__('numpy').nanmean(sils)
avg_conf = float(ensemble.confidence.mean())

verdict = (
    '✓✓ MUY ADECUADO — regímenes muy bien diferenciados' if avg_sil > 0.40 else
    '✓  ADECUADO — regímenes diferenciados'              if avg_sil > 0.28 else
    '~  PARCIALMENTE ADECUADO — separación moderada'     if avg_sil > 0.18 else
    '⚠  POCO ADECUADO — regímenes solapados (activo muy errático)'
)
conf_str = ('✓✓ Alta' if avg_conf>0.80 else '✓  Moderada' if avg_conf>0.60 else '~  Baja')

print(f'╔{"═"*54}╗')
print(f'║  INDICADORES DE ADECUACIÓN — {TICKER:<24}║')
print(f'╠{"═"*54}╣')
print(f'║  {"Modelo":<8}  {"Sil.Train":<24}  {"Sil.Test":<14}║')
print(f'╠{"═"*54}╣')
for mod in ["K-Means","GMM","HMM"]:
    tr = _rate(m.loc[mod,'Sil_Train'])
    te = _rate(m.loc[mod,'Sil_Test'])
    print(f'║  {mod:<8}  {tr:<24}  {te:<14}║')
print(f'╠{"═"*54}╣')
print(f'║  Confianza consenso: {avg_conf:.0%}  {conf_str:<28}║')
print(f'╠{"═"*54}╣')
print(f'║  VEREDICTO: {verdict:<42}║')
print(f'╚{"═"*54}╝')

# Distribución
dist = ensemble.consensus.value_counts(normalize=True).mul(100).round(1)
print('\n  Distribución de régimen:')
for reg, pct in dist.items():
    bar = '█' * int(pct/3)
    print(f'  {reg:>9}: {pct:5.1f}%  {bar}')

# Estado actual
reg_now  = ensemble.consensus.iloc[-1]
conf_now = float(ensemble.confidence.iloc[-1])
str_now  = ensemble.signal_strength.iloc[-1]
emoji    = {'bull':'🟢','bear':'🔴','sideways':'🟡'}.get(reg_now,'⚪')
print(f'\n  Último dato ({ensemble.consensus.index[-1].date()}): {emoji} {reg_now.upper()}  (conf {conf_now:.0%}, {str_now})')

# Retornos forward por régimen
fwd = ensemble.forward_returns
print('\n  Retornos medios por régimen (histórico):')
if not fwd.empty and 'fwd_ret_5d' in fwd.columns:
    try:
        tbl = fwd['fwd_ret_5d']['mean'].rename('mean_5d')
        for c, v in tbl.items():
            print(f'    Cluster {c}: {float(v):+.3%} en 5d')
    except:
        pass

In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  🔍  CAPA 2 — TRIGGER MULTI-TIMEFRAME               ║
# ║      Anomalías diario / semanal / quincenal / mensual║
# ╚══════════════════════════════════════════════════════╝
from src.trigger import run_multitf_trigger, TIMEFRAMES

print(f'\n{"═"*58}')
print(f'  🔍 CAPA 2 — TRIGGER MULTI-TIMEFRAME  ({TICKER})')
print(f'  Umbral: ±{THRESHOLD}σ')
print(f'{"═"*58}\n')

trigger = run_multitf_trigger(
    prices, TICKER,
    timeframes=TIMEFRAMES,
    window=252,
    threshold=THRESHOLD,
)

# Tabla de fuerza de reversión por timeframe
print(f'  {"Timeframe":<22} {"Anomalías":>10} {"P(rev↓ tras ↑)":>16} {"P(rev↑ tras ↓)":>16} {"Fuerza":>12}')
print(f'  {"─"*78}')

reversion_scores = {}
for tf_name, tfr in trigger.timeframes.items():
    n_high = (tfr.df['event_type']=='ANOMALY_HIGH').sum()
    n_low  = (tfr.df['event_type']=='ANOMALY_LOW').sum()
    if not tfr.ttest.empty:
        ht = tfr.ttest[tfr.ttest['Tipo']=='ANOMALY_HIGH']
        lt = tfr.ttest[tfr.ttest['Tipo']=='ANOMALY_LOW']
        h_rev = len(ht[(ht['Sig. (p<0.05)']=='✓')&(ht['Diferencia']<0)])
        l_rev = len(lt[(lt['Sig. (p<0.05)']=='✓')&(lt['Diferencia']>0)])
        p_h = h_rev/max(len(ht),1)
        p_l = l_rev/max(len(lt),1)
    else:
        p_h = p_l = 0.0
    score = p_h + p_l
    reversion_scores[tf_name] = score
    v = ('✓✓ Fuerte'  if score>=1.5 else '✓  Moderada' if score>=0.8 else '~  Débil')
    print(f'  {tf_name:<22} {n_high+n_low:>10} {p_h:>16.0%} {p_l:>16.0%} {v:>12}')

best_tf = max(reversion_scores, key=reversion_scores.get)
best_sc = reversion_scores[best_tf]
print(f'\n  → Timeframe con mayor reversión: {best_tf}')
adj = ('FUERTE' if best_sc>=1.5 else 'MODERADA' if best_sc>=0.8 else 'DÉBIL')
print(f'  → Hipótesis de reversión a la media: {adj} para {TICKER}')

# Rango normal diario
tf0 = list(trigger.timeframes.values())[0]
lo, hi = tf0.normal_range
print(f'\n  Rango normal diario: [{lo:+.2%}  ──  {hi:+.2%}]')
print(f'  Señal BUY_DIP  si retorno diario < {lo*THRESHOLD:.2%}')
print(f'  Señal TAKE_PROFIT si retorno diario > {hi*THRESHOLD:.2%}')

# Anomalías recientes
recent_anom = tf0.anomalies
if not recent_anom.empty:
    print(f'\n  Últimas 5 anomalías detectadas (diario):')
    display(recent_anom[['Retorno','Z-score','Tipo']].tail(5))


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  😱  CAPA 3 — OVERLAY VIX                           ║
# ║      Ajusta régimen con miedo macro                  ║
# ╚══════════════════════════════════════════════════════╝
vix_result = None

if USE_VIX and vix_series is not None:
    from src.vix import apply_vix_overlay, print_vix_summary

    print(f'\n{"═"*58}')
    print(f'  😱 CAPA 3 — OVERLAY VIX  ({TICKER})')
    print(f'{"═"*58}\n')

    vix_result = apply_vix_overlay(
        ensemble.consensus,
        ensemble.confidence,
        vix_series,
    )
    print_vix_summary(vix_result)

    # Comparativa base vs +VIX
    changed = (vix_result.df['Base_Regime'] != vix_result.df['Final_Regime']).sum()
    total   = len(vix_result.df)
    print(f'\n  Días con régimen modificado por VIX: {changed} / {total}  ({changed/total:.1%})')

    # Régimen actual con VIX
    vix_now  = float(vix_result.vix_level.iloc[-1])
    vixr_now = str(vix_result.vix_regime.iloc[-1])
    reg_vix  = vix_result.regime.iloc[-1]
    conf_vix = float(vix_result.confidence.iloc[-1])
    vix_em   = ('😰' if vix_now>35 else '😟' if vix_now>25 else '😐' if vix_now>15 else '😊')
    print(f'\n  VIX actual: {vix_em} {vix_now:.1f}  ({vixr_now})')
    print(f'  Régimen +VIX: {reg_vix.upper()}  (conf: {conf_vix:.0%})')

    # Distribución comparada
    d_base = ensemble.consensus.value_counts(normalize=True).mul(100).round(1)
    d_vix  = vix_result.regime.value_counts(normalize=True).mul(100).round(1)
    print(f'\n  {'Régimen':<12} {'Base':>8} {'+ VIX':>8}  Cambio')
    print(f'  {"─"*36}')
    for reg in ['bull','sideways','bear']:
        b = float(d_base.get(reg, 0))
        v = float(d_vix.get(reg, 0))
        print(f'  {reg:<12} {b:>7.1f}% {v:>7.1f}%  {v-b:+.1f}%')
else:
    print('ℹ️  VIX desactivado (USE_VIX=False o vix_data.csv no encontrado)')
    print('   Puedes activarlo con USE_VIX=True en la celda Config')


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  🎯  CAPA 4 — SEÑAL COMBINADA                       ║
# ║      Trigger + Régimen + VIX → BUY_DIP / TAKE_PROFIT║
# ╚══════════════════════════════════════════════════════╝
from src.combined_signal import run_combined_signal

print(f'\n{"═"*58}')
print(f'  🎯 CAPA 4 — SEÑAL COMBINADA  ({TICKER})')
print(f'{"═"*58}\n')

sig_base = run_combined_signal(
    prices, TICKER,
    ensemble_result=ensemble,
    vix_series=None,
    threshold=THRESHOLD,
    target_horizon=HORIZON,
    min_score=MIN_SCORE,
)

sig_vix = None
if USE_VIX and vix_series is not None:
    sig_vix = run_combined_signal(
        prices, TICKER,
        ensemble_result=ensemble,
        vix_series=vix_series,
        threshold=THRESHOLD,
        target_horizon=HORIZON,
        min_score=MIN_SCORE,
    )

# Tabla comparativa
print(f'  {"Versión":<12} {"TAKE_PROFIT":>12} {"BUY_DIP":>9} {"Activas":>8} {"WR +5d":>8} {"WR +10d":>8}')
print(f'  {"─"*60}')
for sig in [s for s in [sig_base, sig_vix] if s is not None]:
    s = sig.summary
    print(f'  {s["Versión"]:<12} {s["Señales TAKE_PROFIT"]:>12} {s["Señales BUY_DIP"]:>9} '
          f'{s["Señales activas"]:>8} {s.get("Win rate +5d","—"):>8} {s.get("Win rate +10d","—"):>8}')

# Mostrar todas las señales activas
sig_show = sig_vix if sig_vix is not None else sig_base
if not sig_show.active_signals.empty:
    print(f'\n  Señales activas [{sig_show.version}] — todas las señales históricas:')
    cols = ['Fecha','Señal','Puntos','Confianza','Retorno hoy','Z-score',
            'P(reversión)','Precio','Target','Stop Loss','R/R','Potencial','Régimen']
    if 'VIX' in sig_show.active_signals.columns:
        cols.append('VIX')
    display(sig_show.active_signals[cols].reset_index(drop=True))

# Señales de alta confianza
hc = sig_show.active_signals[sig_show.active_signals['Confianza']=='Alta']
if not hc.empty:
    print(f'\n  ★ SEÑALES DE ALTA CONFIANZA (score ≥4/5):')
    for _, row in hc.iterrows():
        em = '🟢' if row['Señal']=='BUY_DIP' else '🔴'
        print(f'  {em} [{row["Fecha"]}]  {row["Señal"]}  score {row["Puntos"]}/5')
        print(f'     ${float(row["Precio"]):.2f}  →  Target ${float(row["Target"]):.2f}  |  Stop ${float(row["Stop Loss"]):.2f}')
        print(f'     {row["Razones"]}')
else:
    print('\n  (Sin señales de alta confianza en este histórico)')


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  💰  P&L vs BUY & HOLD                              ║
# ╚══════════════════════════════════════════════════════╝
from src.combined_signal import run_signal_pnl, print_pnl_comparison

print(f'\n{"═"*58}')
print(f'  💰 P&L vs BUY & HOLD  ({TICKER})')
print(f'{"═"*58}')

pnl_base = run_signal_pnl(prices, sig_base, horizon=HORIZON)
pnl_vix  = run_signal_pnl(prices, sig_vix, horizon=HORIZON) if sig_vix else None

print_pnl_comparison(pnl_base, pnl_vix)

# Tabla visual
rows = list(pnl_base['metrics'])
if pnl_vix:
    rows.insert(1, pnl_vix['metrics'][0])
df_pnl = __import__('pandas').DataFrame(rows).set_index('Estrategia')
display(df_pnl)


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  📊  DASHBOARD VISUAL COMPLETO                      ║
# ╚══════════════════════════════════════════════════════╝
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import numpy as np

REGIME_COLORS = {'bull':'#2ea043','bear':'#f85149','sideways':'#d4ac0d'}

def _shade_regimes(ax, prices, consensus):
    dates = prices.index
    prev_r, start_i = None, 0
    for i, date in enumerate(dates):
        r = consensus.get(date, None)
        if r != prev_r:
            if prev_r and start_i < i:
                ax.axvspan(dates[start_i], dates[i],
                           alpha=0.20, color=REGIME_COLORS.get(prev_r,'gray'), zorder=1)
            prev_r, start_i = r, i
    if prev_r and start_i < len(dates)-1:
        ax.axvspan(dates[start_i], dates[-1],
                   alpha=0.20, color=REGIME_COLORS.get(prev_r,'gray'), zorder=1)

fig = plt.figure(figsize=(18,15))
fig.patch.set_facecolor('#0d1117')
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.3)

ax1 = fig.add_subplot(gs[0, :])   # precio + régimen
ax2 = fig.add_subplot(gs[1, 0])   # z-score
ax3 = fig.add_subplot(gs[1, 1])   # VIX
ax4 = fig.add_subplot(gs[2, 0])   # equity curves
ax5 = fig.add_subplot(gs[2, 1])   # silhouette

for ax in [ax1,ax2,ax3,ax4,ax5]:
    ax.set_facecolor('#161b22')
    ax.tick_params(colors='white')
    for sp in ax.spines.values(): sp.set_color('#30363d')

# ── 1. Precio + régimen + señales ────────────────────
ax1.plot(prices.index, prices.values, color='#58a6ff', lw=1.2, zorder=3)
_shade_regimes(ax1, prices, ensemble.consensus)

sig_show = sig_vix if sig_vix is not None else sig_base
for _, row in sig_show.active_signals.iterrows():
    d = __import__('pandas').Timestamp(row['Fecha'])
    if d in prices.index:
        p = float(prices.loc[d])
        c,m = ('#2ea043','^') if row['Señal']=='BUY_DIP' else ('#f85149','v')
        ax1.scatter(d, p, marker=m, s=90, color=c, zorder=5)

leg = [
    mpatches.Patch(color='#2ea043',alpha=0.5,label='Bull'),
    mpatches.Patch(color='#f85149',alpha=0.5,label='Bear'),
    mpatches.Patch(color='#d4ac0d',alpha=0.5,label='Sideways'),
    Line2D([0],[0],marker='^',color='w',markerfacecolor='#2ea043',ms=8,ls='None',label='BUY_DIP'),
    Line2D([0],[0],marker='v',color='w',markerfacecolor='#f85149',ms=8,ls='None',label='TAKE_PROFIT'),
]
ax1.legend(handles=leg,loc='upper left',facecolor='#161b22',labelcolor='white',fontsize=8,ncol=5)
ax1.set_title(f'LONQ — {TICKER.upper()}  |  Precio + Régimen + Señales',color='white',pad=8)
ax1.set_ylabel('Precio ($)',color='white')
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x:.0f}'))

# ── 2. Z-score ────────────────────────────────────────
tf0 = list(trigger.timeframes.values())[0]
z   = tf0.df['z_score'].dropna()
z   = z.loc[z.index.intersection(prices.index)]
ax2.plot(z.index, z.values, color='#8b949e', lw=0.8, alpha=0.7)
ax2.axhline( THRESHOLD, color='#f85149', lw=1.2, ls='--', label=f'+{THRESHOLD}σ')
ax2.axhline(-THRESHOLD, color='#2ea043', lw=1.2, ls='--', label=f'-{THRESHOLD}σ')
ax2.axhline(0, color='#58a6ff', lw=0.5, alpha=0.4)
anom = tf0.df[tf0.df['event_type']!='NORMAL']
for _,row in anom.iterrows():
    if row.name in z.index:
        c = '#f85149' if row['event_type']=='ANOMALY_HIGH' else '#2ea043'
        ax2.scatter(row.name, row['z_score'], color=c, s=20, zorder=5)
ax2.legend(facecolor='#161b22',labelcolor='white',fontsize=7)
ax2.set_title('Z-Score Diario (Trigger Anomalías)',color='white',pad=6)
ax2.set_ylabel('Z-score',color='white')

# ── 3. VIX ────────────────────────────────────────────
if vix_result is not None:
    vv = vix_result.vix_level.loc[vix_result.vix_level.index.intersection(prices.index)]
    ax3.fill_between(vv.index, vv.values, alpha=0.3, color='#d4ac0d')
    ax3.plot(vv.index, vv.values, color='#d4ac0d', lw=0.9)
    ax3.axhline(15, color='#2ea043', lw=0.8, ls=':', label='VIX 15 (calma)')
    ax3.axhline(25, color='#d4ac0d', lw=1.0, ls='--',label='VIX 25 (alerta)')
    ax3.axhline(35, color='#f85149', lw=1.2, ls='--',label='VIX 35 (pánico)')
    ax3.legend(facecolor='#161b22',labelcolor='white',fontsize=7)
    ax3.set_title('VIX — Índice de Miedo Macro',color='white',pad=6)
    ax3.set_ylabel('VIX',color='white')
else:
    ax3.axis('off')
    ax3.text(0.5,0.5,'VIX desactivado\n(USE_VIX=False)',ha='center',va='center',
             color='#8b949e',fontsize=12,transform=ax3.transAxes)

# ── 4. Equity curves ──────────────────────────────────
ax4.plot(pnl_base['equity_bh'].index, pnl_base['equity_bh'].values,
         color='#8b949e', lw=1.5, label='Buy & Hold', alpha=0.8)
ax4.plot(pnl_base['equity_s'].index,  pnl_base['equity_s'].values,
         color='#58a6ff', lw=1.8, label='Señales [Base]')
if pnl_vix:
    ax4.plot(pnl_vix['equity_s'].index, pnl_vix['equity_s'].values,
             color='#2ea043', lw=1.8, ls='--', label='Señales [+VIX]')
ax4.axhline(1.0, color='white', lw=0.5, alpha=0.3)
ax4.legend(facecolor='#161b22',labelcolor='white',fontsize=8)
ax4.set_title('Curva de Equity vs Buy & Hold',color='white',pad=6)
ax4.set_ylabel('Retorno acumulado',color='white')
ax4.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x:.1f}x'))

# ── 5. Silhouette / Adecuación ────────────────────────
m   = ensemble.metrics
mods = ['K-Means','GMM','HMM']
sil_tr = [float(m.loc[k,'Sil_Train']) for k in mods]
sil_te = [float(m.loc[k,'Sil_Test'])  for k in mods]
x = np.arange(len(mods))
c_fn = lambda s: '#2ea043' if s>0.28 else '#d4ac0d' if s>0.18 else '#f85149'
b1 = ax5.bar(x-0.2, sil_tr, 0.35, label='Train',
             color=[c_fn(s) for s in sil_tr], alpha=0.9)
b2 = ax5.bar(x+0.2, sil_te, 0.35, label='Test',
             color=[c_fn(s) for s in sil_te], alpha=0.55)
ax5.axhline(0.30, color='white', lw=0.8, ls='--', alpha=0.5, label='Umbral bueno')
ax5.set_xticks(x); ax5.set_xticklabels(mods, color='white')
ax5.legend(facecolor='#161b22',labelcolor='white',fontsize=7)
ax5.set_title('Silhouette Score — Adecuación del Ensemble',color='white',pad=6)
ax5.set_ylabel('Silhouette',color='white')
ax5.set_ylim(0, max(0.55, max(sil_tr+sil_te)+0.08))
for bar in list(b1)+list(b2):
    h = bar.get_height()
    if h > 0.01:
        ax5.text(bar.get_x()+bar.get_width()/2, h+0.003, f'{h:.2f}',
                 ha='center', va='bottom', color='white', fontsize=7)

plt.suptitle(f'LONQ — Dashboard Completo: {TICKER.upper()}',
             color='white', fontsize=15, y=0.99, fontweight='bold')
plt.savefig(f'lonq_{TICKER.lower()}_dashboard.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()
print(f'✅ Dashboard guardado: lonq_{TICKER.lower()}_dashboard.png')


In [ ]:
# ╔══════════════════════════════════════════════════════╗
# ║  🖥️  PANEL DE ESTADO ACTUAL                         ║
# ║     ¿Qué está pasando? ¿Cuál es la señal vigente?   ║
# ╚══════════════════════════════════════════════════════╝
from datetime import date as dt_date
import numpy as np

print(f'\n{"═"*62}')
print(f'  🖥️  ESTADO ACTUAL — {TICKER.upper()}')
print(f'  (último dato disponible: {prices.index[-1].date()})')
print(f'{"═"*62}')

# Precio
p_last = float(prices.iloc[-1])
p_prev = float(prices.iloc[-2])
p_chg  = p_last/p_prev - 1
p_em   = '📈' if p_chg>0 else '📉'
print(f'\n  {p_em}  Precio:    ${p_last:.2f}  ({p_chg:+.2%} vs día anterior)')

# Régimen
reg_now  = ensemble.consensus.iloc[-1]
conf_now = float(ensemble.confidence.iloc[-1])
str_now  = ensemble.signal_strength.iloc[-1]
reg_em   = {'bull':'🟢','bear':'🔴','sideways':'🟡'}.get(reg_now,'⚪')
print(f'  {reg_em}  Régimen:   {reg_now.upper():<10}  conf {conf_now:.0%}  ({str_now})')

# VIX
if vix_result is not None:
    vix_now  = float(vix_result.vix_level.iloc[-1])
    vixr_now = str(vix_result.vix_regime.iloc[-1])
    reg_vix  = vix_result.regime.iloc[-1]
    vix_em   = '😰' if vix_now>35 else '😟' if vix_now>25 else '😐' if vix_now>15 else '😊'
    print(f'  {vix_em}  VIX:       {vix_now:.1f}  ({vixr_now})  →  Régimen +VIX: {reg_vix.upper()}')

# Z-score actual
tf0 = list(trigger.timeframes.values())[0]
last_day = prices.index[-1]
if last_day in tf0.df.index:
    z_now = float(tf0.df.loc[last_day,'z_score'])
    anom  = tf0.df.loc[last_day,'event_type']
    z_em  = '🚨' if abs(z_now)>THRESHOLD else '✅'
    print(f'  {z_em}  Z-score:   {z_now:+.2f}σ  ({anom})')

# Señales recientes
print(f'\n  {"─"*60}')
print(f'  SEÑALES RECIENTES (últimas 5):')
sig_show = sig_vix if sig_vix is not None else sig_base
if not sig_show.active_signals.empty:
    for _, row in sig_show.active_signals.tail(5).iterrows():
        em = '🟢' if row['Señal']=='BUY_DIP' else '🔴'
        print(f'  {em} [{row["Fecha"]}]  {row["Señal"]:<14} score {row["Puntos"]}/5  '
              f'{row["Confianza"]:<7}  {row["Retorno hoy"]}  →  Target ${float(row["Target"]):.2f}')
else:
    print('  Sin señales activas en el histórico disponible')

# Niveles de alerta para la próxima sesión
print(f'\n  {"─"*60}')
print(f'  NIVELES DE ALERTA — próxima sesión:')
tf0_df   = tf0.df
sigma_r  = float(tf0_df['ret'].rolling(252,min_periods=30).std().iloc[-1])
mu_r     = float(tf0_df['ret'].rolling(252,min_periods=30).mean().iloc[-1])
dip_ret  = mu_r - THRESHOLD * sigma_r
tp_ret   = mu_r + THRESHOLD * sigma_r
p_dip    = p_last * np.exp(dip_ret)
p_tp     = p_last * np.exp(tp_ret)
print(f'  🟢 BUY_DIP     si cierra por debajo de ${p_dip:.2f}  ({dip_ret:+.2%})')
print(f'  🔴 TAKE_PROFIT si cierra por encima de  ${p_tp:.2f}  ({tp_ret:+.2%})')
print(f'  ✅ HOLD        si cierra entre ${p_dip:.2f} y ${p_tp:.2f}')

print(f'\n{"═"*62}')
print(f'  Análisis LONQ completado ✅')
print(f'  Cambia TICKER en la celda Config y ejecuta todo de nuevo')
print(f'{"═"*62}')
